# Arabic AraBERT experiment — The Turing Test Revisited

This notebook presents Mohammed AlDamegh's Arabic model-training code from a team graduation project. His stated responsibilities were cleaning Arabic data collected by teammates and training the Arabic model. Other team members collected data and worked on English-language components; their data and code are not included here.

This is a sanitized historical research notebook, not a verified production detector. All saved outputs, execution counts, cell metadata, and original notebook metadata have been removed. Data-preview cells have been replaced with comments. An example quotation has been replaced with newly authored synthetic text.

Before running, supply your own authorized CSV files at `../data/train.csv` and `../data/test.csv`, with `text` and `label` columns (`0 = AI`, `1 = Human`), after independently checking the label convention. No original data or trained weights are provided. Review the dependency versions and the limitations in the README first.

The initial model comes from an upstream pretrained checkpoint; its classification head is not a trained detector for this task. The original split and metric calculations are retained for transparency, not endorsed as a validated evaluation protocol. Model downloads contact upstream services. Run only in an isolated research environment with data you are permitted to use.


In [ ]:
import torch
from transformers import AutoTokenizer
from arabert.preprocess import ArabertPreprocessor
# for text classification
from transformers import AutoModelForSequenceClassification

## araBERT Preprocessor

In [ ]:
model_name="bert-base-arabert"
arabert_prep = ArabertPreprocessor(model_name=model_name)

## Initializing Model

In [ ]:
checkpoint = "aubmindlab/bert-base-arabertv02-twitter"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, return_dict=True)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

## Inference

In [ ]:
def classify(text):
    text = arabert_prep.preprocess(text)
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=1)
    return probs

In [ ]:
text = "هذا مثال مصطنع لاختبار واجهة النموذج فقط"
probs = classify(text)
print(probs)


# Training Setup

## Dataset Prep

## Data Preprocessing

In [ ]:

from datasets import load_dataset

dataset = load_dataset("csv", data_files="../data/train.csv", split="train" )

dataset = dataset.train_test_split(test_size=0.03)

In [ ]:
print(dataset)

In [ ]:
# Raw dataset preview omitted from this public notebook.


In [ ]:
# Raw dataset preview omitted from this public notebook.


### Applying araBERT Preprocessor

In [ ]:
def arabic_preprocess(examples):
    return {"text":arabert_prep.preprocess(examples["text"]), "label":examples["label"]}

In [ ]:
# Raw dataset preview omitted from this public notebook.


In [ ]:
dataset = dataset.map(arabic_preprocess)

In [ ]:
# Raw dataset preview omitted from this public notebook.


In [ ]:
# Raw dataset preview omitted from this public notebook.


In [ ]:
def preprocess_function(examples):
    result = tokenizer(examples["text"],truncation=True,   
                       max_length=128, return_overflowing_tokens=True)

    sample_map = result.pop("overflow_to_sample_mapping")
    for key, values in examples.items():
        result[key] = [values[i] for i in sample_map]
    return result
    # return tokenizer(arabic_prep, truncation=True, max_length=512, padding="max_length")

In [ ]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Model Instantiation

In [ ]:
# find number of labels
num_labels = len(set(dataset["train"]["label"]))
print(num_labels)

# get label names
label_names = ["AI", "Human"]
print(label_names)

In [ ]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in id2label.items()}

In [ ]:
print(id2label)
print(label2id)

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id
)

In [ ]:
model.config.hidden_dropout = 0.2

In [ ]:
print(model.config)

## Custom Training Loop

### Hyper parameters

In [ ]:
LEARNING_RATE = 3e-5
BATCH_SIZE = 16

### Data Prep

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")
tokenized_dataset["train"].column_names

In [ ]:
batch_size = BATCH_SIZE
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_dataset["train"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_dataset["test"], shuffle=True, batch_size=batch_size, collate_fn=data_collator
)

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

### Training

In [ ]:
from tqdm.auto import tqdm
# from transformers import AdamW
from transformers import get_scheduler
from torch.utils.tensorboard import SummaryWriter
import evaluate
from torch.optim import AdamW
exp = "5"

writer = SummaryWriter(log_dir="../artifacts/logs/araBERT-base_exp"+exp)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
metric = evaluate.load("glue", "mrpc", device=device)

# optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)


# lr_scheduler
num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))
model.to(device)
for epoch in range(num_epochs):
    # training epoch
    train_loss = 0
    model.train()
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        scores = outputs.logits
        probs = scores.softmax(dim=1)

        predictions = torch.argmax(probs, dim=-1)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        metric.add_batch(predictions=predictions, references=batch["labels"])
        train_loss += loss.item()
        progress_bar.update(1)
        progress_bar.set_postfix(train_loss=loss.item())

    train_out = metric.compute()
    train_accuracy = train_out["accuracy"]
    train_f1 = train_out["f1"]
    train_loss = train_loss / len(train_dataloader)
    print(f"Epoch {epoch} - Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.4f}, f1: {train_f1:.4f}")
    
    # Write to tensorboard
    writer.add_scalar("train/loss", train_loss, epoch)
    writer.add_scalar("train/accuracy", train_accuracy, epoch)
    writer.add_scalar("train/f1", train_f1, epoch)

    # evaluating epoch
    model.eval()
    eval_loss = 0
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        scores = outputs.logits
        probs = scores.softmax(dim=1)
        predictions = torch.argmax(probs, dim=-1)
        eval_loss += loss.item()
        metric.add_batch(predictions=predictions, references=batch["labels"])

    eval_loss = eval_loss / len(eval_dataloader)
    eval_out = metric.compute()
    eval_accuracy = eval_out["accuracy"]
    eval_f1 = eval_out["f1"]
    # eval_accuracy, eval_f1, recall, precision, eval_loss =  evaluate_test_set(model, tokenizer, dataset["test"], batch_size=256)
    print(f"Epoch {epoch} - Eval loss: {eval_loss:.4f}, accuracy: {eval_accuracy:.4f}, f1: {eval_f1:.4f}")

    # Write to tensorboard
    writer.add_scalar("eval/loss", eval_loss, epoch)
    writer.add_scalar("eval/accuracy", eval_accuracy, epoch)
    writer.add_scalar("eval/f1", eval_f1, epoch)

    # progress_bar.update(len(train_dataloader))
    # add text to the progress bar
    progress_bar.set_postfix(epochs = epoch,
        train_loss=train_loss, eval_loss=eval_loss, train_acc=train_accuracy, eval_acc=eval_accuracy
    )

    model.save_pretrained("../artifacts/trained/araBERT-base_exp"+exp+"/checkpoint-"+str((epoch+1)*len(train_dataloader)))
    tokenizer.save_pretrained("../artifacts/trained/araBERT-base_exp"+exp+"/checkpoint-"+str((epoch+1)*len(train_dataloader)))


# close the tensorboard writer
writer.close()

# save the model
# model.save_pretrained("../artifacts/trained/araBERT-base_exp"+exp)
# tokenizer.save_pretrained("../artifacts/trained/araBERT-base_exp"+exp)

## Final Inference

In [ ]:
from datasets import load_dataset
import torch

dataset = load_dataset("csv", data_files="../data/test.csv", split="train")

dataset = dataset.train_test_split(test_size=0.97)

In [ ]:
from transformers import AutoTokenizer
model_path = "../artifacts/model"  # Set to a checkpoint you trained and may use.
tokenizer = AutoTokenizer.from_pretrained(model_path)

text = dataset["train"][0]["text"]

inputs = tokenizer(text, return_tensors="pt")

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs.to(device)

with torch.no_grad():
    logits = model(**inputs).logits

In [ ]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

### Evaluation

In [ ]:
model_name = "bert-base-arabert"
from arabert.preprocess import ArabertPreprocessor
arabic_prep = ArabertPreprocessor(model_name=model_name)
def arabic_preprocess(examples):
    return {"text":arabert_prep.preprocess(examples["text"]), "label":examples["label"]}

In [ ]:
dataset = dataset.map(arabic_preprocess)

In [ ]:
from tqdm.notebook import tqdm
import evaluate
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metric = evaluate.load("glue", "sst2", device=device)
# import precision_recall_fscore_support
def evaluate_test_set(model, tokenizer, test_dataset, batch_size=32, pbar = False):
    if torch.cuda.is_available():
        device = "cuda"
    else :
        device = "cpu"
    print(f"Device: {device}")
    model.to(device)
    model.eval()
    acc = 0
    f1 = 0
    recall = 0
    precision = 0

    predictions = []
    labels = []

    test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)
    if pbar:
        progress = tqdm(enumerate(test_dataloader), total=len(test_dataloader))
    for idx, batch in enumerate(test_dataloader):
        inputs = tokenizer(batch["text"], return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        scores = outputs.logits
        probs = scores.softmax(dim=1)
        predicted_class_ids = probs.argmax(dim=1).tolist()

        predictions.extend(predicted_class_ids)
        batch_labels = batch["label"]
        labels.extend([x.item() for x in batch_labels])
        unique_labels = np.unique(predicted_class_ids)

        # batch_accuracy = sum([1 if label == prediction else 0 for label, prediction in zip(batch_labels, predicted_class_ids)])
        # batch_accuracy = batch_accuracy / len(batch_labels)

        batch_metric = metric.compute(predictions=predicted_class_ids, references=batch_labels)
        batch_acc = batch_metric["accuracy"]
        batch_f1 = f1_score(batch_labels, predicted_class_ids, average='weighted', labels=unique_labels)
        batch_recall = recall_score(batch_labels, predicted_class_ids, average='weighted', labels=unique_labels)
        batch_precision = precision_score(batch_labels, predicted_class_ids, average='weighted', labels=unique_labels)
        # print(batch_accuracy, batch_f1, batch_recall, batch_precision)
    
        acc += batch_metric["accuracy"]
        f1 += batch_f1
        recall += batch_recall
        precision += batch_precision

        if pbar:
            progress.update(1)
            progress.set_description(f"Batch {idx+1}/{len(test_dataloader)}: {batch_acc*100:.2f}%")
        # print(f"Batch {idx+1}/{len(test_dataloader)}: {batch_accuracy}")


    acc = acc / len(test_dataloader)
    f1 = f1 / len(test_dataloader)
    recall = recall / len(test_dataloader)
    precision = precision / len(test_dataloader)
    return acc, f1, recall, precision, predictions, labels


In [ ]:
# Raw dataset preview omitted from this public notebook.


In [ ]:
accuracy, f1, recall, precision, predictions, labels =  evaluate_test_set(model, tokenizer, dataset["test"], batch_size=128, pbar=True)

In [ ]:
print(f"Accuracy: {accuracy*100:.2f}%, F1: {f1*100:.2f}%, Recall: {recall*100:.2f}%, Precision: {precision*100:.2f}%")

In [ ]:
len(predictions), len(labels)
print(set(predictions))
print(set(labels))

In [ ]:
from sklearn.metrics import classification_report
cr = classification_report(labels, predictions, labels=[0, 1], target_names=["AI", "Human"])
print(cr)